In [ ]:
from pathlib import Path
import geopandas as gpd 
import matplotlib.pyplot as plt 

In [2]:
current_directory = Path.cwd()

In [ ]:
#Define of project paths 

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

raw_data_directory = project_root/"data"/"raw"
processed_data_directory = project_root/"data"/"processed"
figures_directory = project_root/"ouputs"/"figures"
   

In [6]:
#Loading spatial data

boundary_file =(raw_data_directory/"greater_accra_boundary.geojson")
station_file = (processed_data_directory/"verified_swap_station_points.geojson")

study_boundary = gpd.read_file(boundary_file)
station_gdf = gpd.read_file(station_file)

In [ ]:
#estimating suitbale metre-based crs(coordinate refernce system )
metric_crs = study_boundary.estimate_utm_crs()

print(metric_crs)

EPSG:32631


In [9]:
#Projecting datasets metre

study_boundary_projected = (study_boundary.to_crs(metric_crs))
stations_projected = (station_gdf.to_crs(metric_crs))

In [10]:
print(study_boundary_projected)
print(stations_projected)

   bbox_west  bbox_south  bbox_east  bbox_north  place_id  osm_type   osm_id  \
0  -0.519705    5.470652   0.672259    6.107615  40402087  relation  1991849   

        lat       lon     class            type  place_rank  importance  \
0  5.810153  0.099524  boundary  administrative           8    0.500795   

  addresstype                  name                 display_name  \
0       state  Greater Accra Region  Greater Accra Region, Ghana   

                                            geometry  
0  POLYGON ((110026.821 631288.431, 110109.332 63...  
  analysis_id source_station_id station_area                landmark  \
0      PUB001       Station 002    Abelemkpe     Near Pan African TV   
1      PUB002       Station 003       Adenta            Housing Down   
2      PUB003       Station 004     Kaneshie           Near Vitamilk   
3      PUB004       Station 005   Kokomlemle         Opposite Auto Z   
4      PUB005       Station 006      Spintex        Adjacent Ecobank   
5      PU

In [ ]:
#Validating if the datasets of the projected is equal
print("Are same:", stations_projected.crs == study_boundary_projected.crs)

Are same: True


In [13]:
#Checking differnence between projected and actual

print("Before Projection:")
print(station_gdf.geometry.head(3))

print("\n After Projection:")
print(stations_projected.geometry.head(3))

Before Projection:
0    POINT (-0.22053 5.61314)
1    POINT (-0.15099 5.70882)
2    POINT (-0.23944 5.57067)
Name: geometry, dtype: geometry

 After Projection:
0    POINT (143157.227 621422.112)
1    POINT (150927.924 631972.819)
2    POINT (141033.083 616731.371)
Name: geometry, dtype: geometry


Create 3 km station buffers - Each station recives 3km /3,000 metre buffer representing the intial straight line coverage area .

In [14]:
coverage_radius_metre = 3_000

In [ ]:
#creating buffer geodataframe

coverage_buffers = stations_projected.copy()
coverage_buffers['geometry'] = (stations_projected.geometry.buffer(coverage_radius_metre))

In [ ]:
#calculating of buffer areas

coverage_buffers["buffer_area_km2"] = (coverage_buffers.geometry.area/ 1_000_000)